## part a: Derivation of the Option Pricing Formula

The terminal payoff of the European option is

$$
V(T,S_T^{(1)},S_T^{(2)})
=
(S_T^{(1)}-S_T^{(2)})^2.
$$

Under the risk-neutral measure $Q$, both stocks have drift $r$. Their dynamics are

$$
\frac{dS_t^{(1)}}{S_t^{(1)}}=r\,dt+\sigma_1\,dW_t^{(1)},
$$

$$
\frac{dS_t^{(2)}}{S_t^{(2)}}
=r\,dt+\sigma_2\left(\rho\,dW_t^{(1)}
+\sqrt{1-\rho^2}\,dW_t^{(2)}\right).
$$

Let $\tau=T-t$. The risk-neutral pricing formula is

$$
V(t,s_1,s_2)
=e^{-r\tau}E^Q\left[
(S_T^{(1)}-S_T^{(2)})^2
\mid S_t^{(1)}=s_1,S_t^{(2)}=s_2
\right].
$$

**Step 1: Expand the squared payoff**

$$
(S_T^{(1)}-S_T^{(2)})^2
=(S_T^{(1)})^2
-2S_T^{(1)}S_T^{(2)}
+(S_T^{(2)})^2.
$$

Therefore,

$$
V(t,s_1,s_2)
=e^{-r\tau}\left[
E^Q[(S_T^{(1)})^2]
-2E^Q[S_T^{(1)}S_T^{(2)}]
+E^Q[(S_T^{(2)})^2]
\right],
$$

where all expectations are conditional on the stock prices at time $t$.

**Step 2: Calculate the individual second moments**

Under geometric Brownian motion,

$$
S_T^{(i)}
=s_i\exp\left[
\left(r-\frac{\sigma_i^2}{2}\right)\tau
+\sigma_i\sqrt{\tau}Z_i
\right],
$$

where $Z_i$ is standard normal.

Using the moment-generating function of a standard normal random variable,

$$
E[e^{aZ}]=e^{a^2/2},
$$

we obtain

$$
E^Q[(S_T^{(1)})^2]
=s_1^2e^{(2r+\sigma_1^2)\tau},
$$

and similarly,

$$
E^Q[(S_T^{(2)})^2]
=s_2^2e^{(2r+\sigma_2^2)\tau}.
$$

**Step 3: Calculate the cross moment**

Because the two stock returns have correlation $\rho$, the standardized Brownian shocks satisfy

$$
\operatorname{Corr}(Z_1,Z_2)=\rho.
$$

The product of the terminal stock prices is

$$
S_T^{(1)}S_T^{(2)}
=s_1s_2\exp\left[
\left(2r-\frac{\sigma_1^2+\sigma_2^2}{2}\right)\tau
+\sqrt{\tau}(\sigma_1Z_1+\sigma_2Z_2)
\right].
$$

Since

$$
\operatorname{Var}(\sigma_1Z_1+\sigma_2Z_2)
=\sigma_1^2+\sigma_2^2+2\rho\sigma_1\sigma_2,
$$

we have

$$
E^Q[S_T^{(1)}S_T^{(2)}]
=s_1s_2e^{(2r+\rho\sigma_1\sigma_2)\tau}.
$$

**Step 4: Substitute into the pricing formula**

Substituting the three expectations gives

$$
V(t,s_1,s_2)
=e^{-r\tau}\left[
s_1^2e^{(2r+\sigma_1^2)\tau}
-2s_1s_2e^{(2r+\rho\sigma_1\sigma_2)\tau}
+s_2^2e^{(2r+\sigma_2^2)\tau}
\right].
$$

Factoring out $e^{2r\tau}$ and simplifying,

$$
\boxed{
V(t,s_1,s_2;\sigma,\rho)
=e^{r(T-t)}
\left[
s_1^2e^{\sigma_1^2(T-t)}
-2s_1s_2e^{\rho\sigma_1\sigma_2(T-t)}
+s_2^2e^{\sigma_2^2(T-t)}
\right]
}
$$

This is the required arbitrage-free pricing formula.


In [1]:
import numpy as np
import pandas as pd
from itertools import product

# Initial market parameters
r = 0.0132
T = 0.25
dt = 1 / 252
horizon = 5 * dt

S1, S2 = 158.12, 170.33
sig1, sig2 = 0.2214, 0.3036
rho = 0.237

# European option pricing formula from Part (a)
def option_price(s1, s2, vol1, vol2, corr, tau):
    return np.exp(r * tau) * (
        s1**2 * np.exp(vol1**2 * tau)
        - 2 * s1 * s2 * np.exp(corr * vol1 * vol2 * tau)
        + s2**2 * np.exp(vol2**2 * tau)
    )

# Initial option delta
factor = np.exp(r * T)

delta1 = 2 * factor * (
    S1 * np.exp(sig1**2 * T)
    - S2 * np.exp(rho * sig1 * sig2 * T)
)

delta2 = 2 * factor * (
    S2 * np.exp(sig2**2 * T)
    - S1 * np.exp(rho * sig1 * sig2 * T)
)

# Initial delta-hedged portfolio value
V0 = option_price(S1, S2, sig1, sig2, rho, T)
P0 = V0 - delta1 * S1 - delta2 * S2

print("Initial option price:", V0)
print("Delta 1:", delta1)
print("Delta 2:", delta2)
print("Initial portfolio value:", P0)


Initial option price: 921.739865813417
Delta 1: -21.952542517420042
Delta 2: 31.201877264611575
Initial portfolio value: -921.7398658134152


In [2]:
returns = [-0.20, -0.10, -0.05, 0.05, 0.10, 0.20]
vol_ratios = [0.5, 0.75, 1.25, 1.5, 1.75, 2.0]
corr_changes = [-0.5, 0, 0.5]

results = []

for x1, x2, m1, m2, dc in product(
    returns, returns, vol_ratios, vol_ratios, corr_changes
):
    # Stock prices after 5 days
    s1_new = S1 * np.exp(x1)
    s2_new = S2 * np.exp(x2)

    # Updated volatility and correlation
    vol1_new = sig1 * m1
    vol2_new = sig2 * m2
    corr_new = rho + dc

    # Reprice the option after 5 days
    V_new = option_price(
        s1_new, s2_new,
        vol1_new, vol2_new,
        corr_new, T - horizon
    )

    # Original hedge shares remain unchanged
    P_new = V_new - delta1 * s1_new - delta2 * s2_new

    loss = P0 - P_new

    results.append([x1, x2, m1, m2, dc, loss])

df = pd.DataFrame(results, columns=[
    "X1", "X2", "Vol1 Ratio", "Vol2 Ratio",
    "Correlation Change", "Loss"
])

print("Number of scenarios:", len(df))
display(df.head())

worst_b = df.loc[df["Loss"].idxmax()]
print("\nWorst-case scenario (b):")
print(worst_b)


Number of scenarios: 3888


,X1,X2,Vol1 Ratio,Vol2 Ratio,Correlation Change,Loss
0,-0.2,-0.2,0.5,0.50,-0.5,299.304282
1,-0.2,-0.2,0.5,0.50,0.0,369.337594
2,-0.2,-0.2,0.5,0.50,0.5,439.506469
3,-0.2,-0.2,0.5,0.75,-0.5,150.468650
4,-0.2,-0.2,0.5,0.75,0.0,255.515997



Worst-case scenario (b):
X1                       0.200000
X2                       0.200000
Vol1 Ratio               0.500000
Vol2 Ratio               0.500000
Correlation Change       0.500000
Loss                  1000.361717
Name: 3782, dtype: float64


### part b: Worst-Case Scenario Risk Measure

We evaluate all 3,888 scenarios and calculate the full portfolio loss using

$$
L_n = -(\Pi_{t+5\Delta}(x_n)-\Pi_t).
$$

The worst-case scenario risk measure is

$$
\rho(L)=\max_{n=1,\ldots,3888}L_n.
$$

The maximum portfolio loss is

$$
\boxed{\rho(L)=1000.3617}.
$$

This occurs under the following scenario:

- Stock 1 log return: +20%
- Stock 2 log return: +20%
- Stock 1 volatility multiplier: 0.50
- Stock 2 volatility multiplier: 0.50
- Correlation change: +0.50

Although the portfolio is initially delta-hedged, the hedge remains fixed over the five-day horizon. Therefore, the portfolio is still exposed to nonlinear price movements and changes in volatility and correlation.

In this scenario, both stocks increase significantly, while their volatilities decrease and their correlation increases. These changes reduce the option's value relative to the fixed hedge position, resulting in the largest portfolio loss.


In [4]:
return_weights = {
    -0.20: 0.5, 0.20: 0.5,
    -0.10: 0.75, 0.10: 0.75,
    -0.05: 1.0, 0.05: 1.0
}

vol_weights = {
    0.5: 0.5, 2.0: 0.5,
    0.75: 1.25, 1.75: 1.25,
    1.25: 0.75, 1.5: 0.75
}

corr_weights = {
    -0.5: 1.25,
    0: 0.5,
    0.5: 1.25
}

df["Weight"] = (
    df["X1"].map(return_weights)
    * df["X2"].map(return_weights)
    * df["Vol1 Ratio"].map(vol_weights)
    * df["Vol2 Ratio"].map(vol_weights)
    * df["Correlation Change"].map(corr_weights)
)

df["Weighted Loss"] = df["Weight"] * df["Loss"]

worst_c = df.loc[df["Weighted Loss"].idxmax()]

print("Worst weighted-loss scenario (c):")
print(worst_c)


Worst weighted-loss scenario (c):
X1                       0.050000
X2                       0.050000
Vol1 Ratio               0.750000
Vol2 Ratio               0.750000
Correlation Change       0.500000
Loss                   672.658388
Weight                   1.953125
Weighted Loss         1313.785913
Name: 2291, dtype: float64


### part c: Weighted Scenario Risk Measure

We assign a weight to each scenario based on the specified risk-factor weights.

The total scenario weight is the product of the five individual weights:

$$
w_n=w_{X_1}w_{X_2}w_{\sigma_1}w_{\sigma_2}w_{\rho}.
$$

The weighted scenario risk measure is

$$
\rho(L)=\max_{n=1,\ldots,3888}\{w_nL_n\}.
$$

The maximum weighted loss is

$$
\boxed{\rho(L)=1313.7859}.
$$

This occurs under the following scenario:

- Stock 1 log return: +5%
- Stock 2 log return: +5%
- Stock 1 volatility multiplier: 0.75
- Stock 2 volatility multiplier: 0.75
- Correlation change: +0.50

The corresponding scenario weight is

$$
w_n=1\times1\times1.25\times1.25\times1.25
=1.953125.
$$

The actual portfolio loss under this scenario is

$$
\boxed{L_n=672.6584}.
$$

Therefore, the weighted loss is

$$
w_nL_n
=1.953125\times672.6584
\approx1313.7859.
$$

Unlike Part (b), the maximum weighted loss does not necessarily occur in the scenario with the largest actual loss.

The weighting scheme assigns greater importance to certain scenarios. Consequently, a scenario with a smaller actual loss can produce a larger weighted risk measure.
